# Week 7 · 05 Paper practice: transfer learning, autoencoders, VAE, anomaly detection

Exam-style practice for the week in the exam's three forms: computing by hand, writing code by hand, and reading PyTorch code. Part II's code is booklet level: two blocks from your booklet (k-means, chapter 7; the MLP forward pass, chapter 12) and three of this week's blocks. About 75 minutes on paper; then type Part II and run the checks. Since the exam is scored now, treat this one as a real test and log the score.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from figures import *
import checks; checks.load(globals(), "05")      # defines check(n)

## M. How to use this notebook

Set a timer for 75 minutes and work on paper: Part I (by hand, about 35 minutes), Part II (code by hand, about 25 minutes), Part III (reading PyTorch, about 15 minutes). Then mark yourself, type Part II into the cells and run the checks. Part IV (Weeks 5–6, spaced) is extra. Log the score and the misses **by topic**.

| Part | What | Points |
|---|---|---|
| I | by hand: parameter counts, transposed convolutions, the reparameterization trick, KL, Mahalanobis, precision/recall, AUC, transfer | 8 × 5 = 40 |
| II | handwritten code (booklet level), then checked | 5 × 8 = 40 |
| III | reading PyTorch code | 5 × 4 = 20 |
| IV | Weeks 5–6, spaced (not scored) |  |

## I. By hand (40 points)

**Drill (on paper).** **I1 · An autoencoder.** Encoder `Linear(784, 256) → ReLU → Linear(256, 16)`, mirrored decoder `Linear(16, 256) → ReLU → Linear(256, 784) → Sigmoid`. (a) The number of parameters of each layer and the total. (b) If all activations were removed and the data centred, what would the best encoder–decoder pair compute?

<details><summary>Worked answer</summary>

(a) $784 \cdot 256 + 256 = 200{,}960$; $256 \cdot 16 + 16 = 4{,}112$; $16 \cdot 256 + 256 = 4{,}352$; $256 \cdot 784 + 784 = 201{,}488$; total $410{,}912$. (b) The projection onto the top-16 principal components of the data: a linear autoencoder with squared error learns PCA's subspace (its codes are PCA's scores up to an invertible 16×16 change of coordinates). (`02` §2)

</details>

**Drill (on paper).** **I2 · Transposed convolutions.** (a) `ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1)` on `(N, 64, 7, 7)`, then `ConvTranspose2d(32, 1, 4, stride=2, padding=1)`: the shapes. (b) A transposed convolution with stride 1 and kernel $\begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}$ on the input $\begin{pmatrix} 1 & 2 \end{pmatrix}$ (1×2): the output.

<details><summary>Worked answer</summary>

(a) $(7 - 1) \cdot 2 - 2 + 3 + 1 = 14$: `(N, 32, 14, 14)`; then $(14 - 1) \cdot 2 - 2 + 4 = 28$: `(N, 1, 28, 28)`. (b) Size $(1 - 1) + 2 = 2$ by $(2 - 1) + 2 = 3$. The 1 stamps $\begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}$ at columns 0–1, the 2 stamps $\begin{pmatrix} 2 & 2 \\ 2 & 2 \end{pmatrix}$ at columns 1–2; overlaps add: $\begin{pmatrix} 1 & 3 & 2 \\ 1 & 3 & 2 \end{pmatrix}$.

</details>

**Drill (on paper).** **I3 · Reparameterization.** The encoder outputs $\mu = (0.5, -1)$ and $\log \sigma^2 = (0, \ln 4)$; this step $\varepsilon = (1, -0.5)$. (a) $\sigma$ and $z$. (b) For the loss $L = z_1 + z_2^2$: $\partial L / \partial \mu$ and $\partial L / \partial \sigma$.

<details><summary>Worked answer</summary>

(a) $\sigma = e^{\frac12 \log\sigma^2} = (1, 2)$; $z = \mu + \sigma \varepsilon = (0.5 + 1,\; -1 + 2 \cdot (-0.5)) = (1.5, -2)$. (b) $\partial L/\partial z = (1,\; 2 z_2) = (1, -4)$. Since $\partial z/\partial \mu = 1$: $\partial L/\partial \mu = (1, -4)$. Since $\partial z/\partial \sigma = \varepsilon$: $\partial L/\partial \sigma = (1 \cdot 1,\; -4 \cdot (-0.5)) = (1, 2)$.

</details>

**Drill (on paper).** **I4 · The KL term.** A 2-dimensional code with $\mu = (1, 0)$ and $\sigma^2 = (1, 0.25)$. Compute $D_{\text{KL}}(\mathcal{N}(\mu, \sigma^2) \,\Vert\, \mathcal{N}(0, I))$ (use $\ln 0.25 = -1.386$). Which coordinate is penalized for its centre and which for its spread?

<details><summary>Worked answer</summary>

Per coordinate $\frac12(\sigma^2 + \mu^2 - 1 - \ln\sigma^2)$. First: $\frac12(1 + 1 - 1 - 0) = 0.5$ (its centre is 1 away from 0). Second: $\frac12(0.25 + 0 - 1 + 1.386) = 0.318$ (its spread is too narrow). Total $0.818$.

</details>

**Drill (on paper).** **I5 · Mahalanobis.** Normal data has mean $(0, 0)$ and covariance $S = \begin{pmatrix} 1 & 0.5 \\ 0.5 & 1 \end{pmatrix}$. Compute $d_M$ for $a = (1, -1)$ and $b = (1, 1)$. Their Euclidean distances are equal; which point is more unusual, and why?

<details><summary>Worked answer</summary>

$\det S = 0.75$, $S^{-1} = \frac{1}{0.75}\begin{pmatrix} 1 & -0.5 \\ -0.5 & 1 \end{pmatrix}$. $a$: $\frac{1}{0.75}(1 + 2 \cdot (-0.5)(1)(-1) + 1) = \frac{3}{0.75} = 4$, so $d_M = 2$. $b$: $\frac{1}{0.75}(1 - 1 + 1) = 1.333$, so $d_M = 1.155$. Both are $\sqrt 2$ from the mean, but $a$ goes **against** the positive correlation (one up, one down), which the data rarely does.

</details>

**Drill (on paper).** **I6 · Precision and recall.** 1,000 parts, 20 of them faulty. A detector flags 50 parts, 15 of which are faulty. TP, FP, FN, TN; precision, recall, false positive rate, F1. Is the 3.6% false positive rate good news?

<details><summary>Worked answer</summary>

TP 15, FP 35, FN 5, TN 945. Precision $15/50 = 0.30$; recall $15/20 = 0.75$; FPR $35/980 = 0.036$; F1 $= \frac{2 \cdot 0.30 \cdot 0.75}{0.30 + 0.75} = 0.429$. The FPR looks small only because it's divided by the 980 good parts; for the inspector, 7 out of 10 alarms are false (precision 0.30). That's why precision–recall is the honest view when anomalies are rare.

</details>

**Drill (on paper).** **I7 · AUC and AP by hand.** Anomalies score $0.9, 0.6, 0.3$; normal points $0.8, 0.5, 0.2, 0.1$. (a) ROC-AUC by counting pairs. (b) Average precision.

<details><summary>Worked answer</summary>

(a) 0.9 beats all 4 normal scores; 0.6 beats 3 (0.5, 0.2, 0.1); 0.3 beats 2. $9/12 = 0.75$. (b) Sorted: 0.9 (A), 0.8, 0.6 (A), 0.5, 0.3 (A), 0.2, 0.1. Precision at each anomaly: $1/1$, $2/3$, $3/5$. AP $= (1 + 0.667 + 0.6)/3 = 0.756$.

</details>

**Drill (on paper).** **I8 · Transfer learning.** A pretrained CNN: `Conv2d(3, 32, 3)`, `Conv2d(32, 64, 3)`, then `Linear(1024, 1000)` for ImageNet's 1,000 classes. For a 5-class task: (a) with feature extraction, how many weights train? (b) Why fine-tune with a learning rate 10 times smaller than for training from scratch?

<details><summary>Worked answer</summary>

(a) A new head `Linear(1024, 5)`: $1024 \cdot 5 + 5 = 5{,}125$; everything else is frozen. (b) The copied weights are already good; large steps driven by a small dataset would overwrite what the large source taught (catastrophic forgetting) and overfit. Small steps keep the weights near the good start and only adjust them.

</details>

## II. Handwritten code (40 points)

Exercises 1–2 are booklet blocks (chapters 7 and 12), from memory. Exercises 3–5 are this week's. Write each on paper first; then type it and run its check.

**Exercise 1 · Booklet 7.4: k-means.** `kmeans(X, k, n_iter=100, seed=0)`: start from `k` random data points (`rng = np.random.default_rng(seed)`, `X[rng.choice(len(X), size=k, replace=False)]`); repeat: assign every point to its nearest centre, move each centre to the mean of its points, stop when the centres stop moving (`np.allclose`). Return `(mu, c, J)` with $J = \sum_i \lVert x_i - \mu_{c(i)}\rVert^2$.

**Example**: what goes in, and what should come out:
```python
X = np.array([[0.0, 0.0], [0.0, 1.0], [10.0, 0.0], [10.0, 1.0]])
[kmeans(X, 2, seed=s)[2] for s in range(3)]    # → [100.0, 1.0, 1.0] as floats
# seed 0 starts with both centres on the left, splits the points top/bottom and gets stuck (J = 100);
# seeds 1 and 2 find the two real clusters (J = 1). That is why booklet 7.4 keeps the best of 5 starts.
```

In [ ]:
def kmeans(X, k, n_iter=100, seed=0):
    pass  # your code

In [ ]:
check(1)

**Exercise 2 · Booklet 12.3: the MLP forward pass.** `X` `(n, d)`, `W1` `(d, h)`, `b1` `(h,)`, `W2` `(h, C)`, `b2` `(C,)`, all tensors. Return the logits $\sigma(X W_1 + b_1) W_2 + b_2$, shape `(n, C)`.

**Example**: what goes in, and what should come out:
```python
mlp_forward(torch.zeros(1, 2), torch.ones(2, 3), torch.zeros(3), torch.ones(3, 2), torch.zeros(2))    # → tensor([[1.5000, 1.5000]])   3 × σ(0)
```

In [ ]:
def mlp_forward(X, W1, b1, W2, b2):
    pass  # your code

In [ ]:
check(2)

**Exercise 3 · Autoencoder flags.** Reconstruction error per row (mean squared error over dim 1, with `no_grad`) for `X_train` and for `X`; threshold = the `q` quantile of the training errors; return a boolean NumPy array: which rows of `X` score above it.

**Example**: what goes in, and what should come out:
```python
Z_tr, good, scratched, others = zeros_data(); m = load_ae_zeros()
ae_flags(m, Z_tr, scratched, 0.99).mean()    # → 0.42   04 §E
```

In [ ]:
def ae_flags(model, X_train, X, q):
    pass  # your code

In [ ]:
check(3)

**Exercise 4 · Freeze the first layers.** `model` is an `nn.Sequential`. Freeze the parameters of its first `n` layers (`model[:n]`), leave the rest trainable, and return the number of trainable weights.

**Example**: what goes in, and what should come out:
```python
freeze_first(nn.Sequential(nn.Linear(4, 4), nn.ReLU(), nn.Linear(4, 2)), 2)    # → 10   only the last Linear: 4·2 + 2
```

In [ ]:
def freeze_first(model, n):
    pass  # your code

In [ ]:
check(4)

**Exercise 5 · ROC-AUC by counting pairs.** `y` 0/1, `s` scores. Return the fraction of (anomaly, normal) pairs where the anomaly scores higher; a tie counts as half. Vectorized: compare `pos[:, None]` with `neg[None, :]`.

**Example**: what goes in, and what should come out:
```python
auc_pairs(np.array([1, 0, 1, 0, 1, 0, 0]), np.array([0.9, 0.8, 0.6, 0.5, 0.3, 0.2, 0.1]))    # → 0.75   I7 (a)
```

In [ ]:
def auc_pairs(y, s):
    pass  # your code

In [ ]:
check(5)

In [ ]:
for n in range(1, 6):
    check(n)

## III. Reading PyTorch code (20 points)

**Drill (on paper).** **III1 · Shapes in a VAE.** `model = VAE(784, 400, 20)` (`03`'s class: `encoder` to 400, two heads to 20, decoder back) and a batch `x` of shape `(64, 784)`. `x_hat, mu, logvar = model(x)`. The shapes of `mu`, `logvar`, the sampled `z` and `x_hat`? How many parameters does `to_logvar` have?

<details><summary>Worked answer</summary>

`mu`, `logvar` and `z`: `(64, 20)`; `x_hat`: `(64, 784)`. `to_logvar = nn.Linear(400, 20)`: $400 \cdot 20 + 20 = 8{,}020$.

</details>

**Drill (on paper).** **III2 · Find the bug.**\n```python\ndef forward(self, x):\n    h = self.encoder(x)\n    mu, logvar = self.to_mu(h), self.to_logvar(h)\n    z = mu + logvar * torch.randn_like(mu)\n    return self.decoder(z), mu, logvar\n```

<details><summary>Worked answer</summary>

`logvar` is $\log\sigma^2$, not $\sigma$: it can be negative, and it's on the wrong scale. The standard deviation is `torch.exp(0.5 * logvar)`: `z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)`.

</details>

**Drill (on paper).** **III3 · Why doesn't the new head learn?**\n```python\nmodel = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)\nfor p in model.parameters():\n    p.requires_grad = False\nopt = torch.optim.Adam(model.parameters(), lr=1e-3)\nmodel.fc = nn.Linear(512, 5)\n```

<details><summary>Worked answer</summary>

The optimizer was created **before** the new head existed, so it holds the old (frozen) parameters and not the new `fc`. Its `step()` never touches the head, which stays random. Create the optimizer after replacing the head, with `model.fc.parameters()`.

</details>

**Drill (on paper).** **III4 · One number instead of many.**\n```python\nwith torch.no_grad():\n    scores = ((model(X_test) - X_test) ** 2).mean()\nflags = scores > threshold\n```\nWhat's wrong, and what's the fix?

<details><summary>Worked answer</summary>

`.mean()` with no `dim` averages over **all** images and pixels: one number for the whole test set, so either everything is flagged or nothing. Use `.mean(dim=1)`: one score per image, shape `(N,)`.

</details>

**Drill (on paper).** **III5 · A denoising autoencoder that only copies.**\n```python\nnoisy = (xb + 0.5 * torch.randn_like(xb)).clamp(0, 1)\nloss = loss_fn(model(noisy), noisy)\n```

<details><summary>Worked answer</summary>

The target is the noisy image, so the network is rewarded for reproducing the noise: it's an ordinary autoencoder on noisy data. The target must be the **clean** image: `loss_fn(model(noisy), xb)`.

</details>

## IV. Weeks 5–6, spaced (not scored)

**Drill (on paper).** **IV1 · Output size.** `Conv2d(16, 32, 3, stride=2, padding=1)` on `(N, 16, 15, 15)`?

<details><summary>Worked answer</summary>

$\lfloor (15 + 2 - 3)/2 \rfloor + 1 = 8$: `(N, 32, 8, 8)`.

</details>

**Drill (on paper).** **IV2 · Adam's first step.** $g_1 = 0.4$, $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\eta = 0.01$, $w_0 = 1$. $m_1$, $s_1$, the bias-corrected $\hat m_1$, $\hat s_1$, and $w_1$ (ignore $\varepsilon$)?

<details><summary>Worked answer</summary>

$m_1 = 0.1 \cdot 0.4 = 0.04$; $s_1 = 0.001 \cdot 0.16 = 0.00016$; $\hat m_1 = 0.04/0.1 = 0.4$; $\hat s_1 = 0.00016/0.001 = 0.16$; $w_1 = 1 - 0.01 \cdot 0.4/\sqrt{0.16} = 1 - 0.01 = 0.99$. Adam's first step is always $\eta$ in size (times the sign of the gradient).

</details>

**Drill (on paper).** **IV3 · Backprop through softmax + cross-entropy.** Logits $z = (2, 0)$, true class 1. $\partial L/\partial z$?

<details><summary>Worked answer</summary>

$p = \text{softmax}(z) = (e^2, 1)/(e^2 + 1) = (0.881, 0.119)$; $y = (0, 1)$; $\partial L/\partial z = p - y = (0.881, -0.881)$.

</details>

---
Mark Parts I–III, then log the score and every miss by topic. Solutions for Part II: `solutions/05_solutions.py`. Next: `06_sunday_quiz.ipynb`.